# 00 - Setup: what am I running on?

**What this step does.** Checks that the machine can actually run the rest of
the project: the right packages import, a compute device is found, randomness
is pinned, and the folders we write to exist.

**Why it exists.** Every number this project reports has to be reproducible.
That is only true if three things hold: the *same package versions*, the
*same random seeds*, and a *record of which configuration produced which
result*. This notebook establishes all three before any modelling happens.
It also makes the project portable - the same code has to run on a laptop
CPU, on a laptop GPU, and on a Kaggle T4, without editing paths by hand.

**What could go wrong.**
- **PyTorch installed without CUDA.** Everything still runs, just 10-50x
  slower. The device report below says plainly which one you got.
- **Kernel pointing at the wrong Python.** Jupyter happily runs a kernel from
  a different environment than the one you installed into. The report prints
  `sys.executable` so you can check.
- **Forgetting to restart the kernel after installing.** Python caches
  imports; a package installed mid-session may not be importable until the
  kernel restarts.
- **Unseeded randomness.** The most dangerous failure, because nothing looks
  broken - you just get different numbers every run and cannot tell a real
  effect from noise.

## 1. Finding the project code

Notebooks live in `notebooks/`, but the code lives in `src/`. Python will not
find `src` unless the *project root* is on the import path.

Rather than hard-coding a path (which breaks the moment you move the folder,
or open it on Kaggle), we walk up from the current directory until we find the
folder that contains `src/config.py`. That works from any working directory.

In [1]:
import sys
import pathlib


def find_project_root(start=None):
    """Walk upward until we find the folder containing src/config.py."""
    here = pathlib.Path(start or pathlib.Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate
    raise RuntimeError(
        f"Could not find the project root above {here}. "
        "Open this notebook from inside the OmniGraph folder."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("project root :", PROJECT_ROOT)
print("on sys.path  :", str(PROJECT_ROOT) in sys.path)

project root : D:\documents\my projects\afterPFE\OmniGraph
on sys.path  : True


## 2. Installing dependencies

On a laptop you install once from `requirements.txt` and never touch this
cell. On **Kaggle or Colab** the machine is fresh every session, so the cell
below installs what is missing.

It is deliberately *conditional*: it imports first and only installs what
actually fails. Reinstalling PyTorch on Kaggle would waste several minutes and
risks replacing the CUDA build with a CPU one.

In [2]:
import importlib
import subprocess
import sys

# package import name -> pip requirement string
WANTED = {
    "torch_geometric": "torch_geometric==2.8.0.post1",
    "sklearn": "scikit-learn>=1.5",
    "networkx": "networkx>=3.3",
    "pandas": "pandas>=2.2",
    "matplotlib": "matplotlib>=3.8",
    "seaborn": "seaborn>=0.13",
    "tqdm": "tqdm>=4.66",
}

missing = []
for import_name, requirement in WANTED.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing.append(requirement)

if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("done - RESTART THE KERNEL, then re-run from the top.")
else:
    print("nothing to install: every dependency already imports.")

# NOTE on torch itself: Kaggle and Colab ship a CUDA build of torch already,
# so we never pip-install torch there. Locally it comes from requirements.txt.

nothing to install: every dependency already imports.


## 3. The environment report

This is the single cell to screenshot when something behaves oddly. It answers:
which interpreter, which versions, which device, and where data will be written.

`src/env.py` does the work; the notebook only prints it.

In [3]:
from src import env

report = env.print_report()


Interpreter
-----------
  python       : 3.12.0
  executable   : D:\documents\my projects\afterPFE\OmniGraph\.venv\Scripts\python.exe
  platform     : Windows-11-10.0.26200-SP0
  environment  : local

Paths
-----
  project root : D:\documents\my projects\afterPFE\OmniGraph
  data root    : D:\documents\my projects\afterPFE\OmniGraph\data  (exists=True)
  runs.jsonl   : D:\documents\my projects\afterPFE\OmniGraph\results\runs.jsonl

Packages
--------
  [ok  ] torch              2.9.1+cu126
  [ok  ] torch_geometric    2.8.0.post1
  [ok  ] numpy              2.5.3
  [ok  ] scipy              1.16.2
  [ok  ] sklearn            1.7.2
  [ok  ] networkx           3.5
  [ok  ] pandas             2.3.3
  [ok  ] matplotlib         3.10.6

Compute device
--------------
  torch        : 2.9.1+cu126  (cuda build: 12.6)
  cuda available: True
  SELECTED     : cuda
  gpu          : NVIDIA GeForce RTX 2050
  gpu memory   : 4.0 GB
  capability   : sm_86


Now assert it. If a package is missing this raises immediately, which is much
better than discovering it three notebooks later inside a training loop.

In [4]:
env.assert_environment_ok(report)


Environment OK: every required package imports cleanly.


## 4. Seeding - why this is not optional

A neural network has several sources of randomness: the initial weights, the
order examples are visited, dropout masks, and (for us) which nodes get
corrupted during self-supervised pretraining.

If those are not pinned, two runs of *identical* code give different numbers.
That matters enormously here, because the whole project is a comparison
between arms. If arm A beats arm B by 1.5 points but reruns of arm A vary by
3 points, the comparison means nothing. Seeding is what lets us tell a real
difference from noise.

`set_seed` pins all four RNGs at once: Python's `random`, NumPy, PyTorch on
CPU, and PyTorch on GPU.

In [5]:
import torch

from src.seeding import set_seed

set_seed(42)
a = torch.randn(4)

set_seed(42)
b = torch.randn(4)

set_seed(7)
c = torch.randn(4)

print("seed 42, draw 1 :", a.tolist())
print("seed 42, draw 2 :", b.tolist())
print("seed  7, draw 1 :", c.tolist())
print()
print("same seed  -> identical?", torch.equal(a, b))
print("other seed -> different?", not torch.equal(a, c))

assert torch.equal(a, b), "seeding is broken: same seed gave different numbers"
assert not torch.equal(a, c), "suspicious: different seeds gave identical numbers"
print("\nSeeding works.")

seed 42, draw 1 : [0.33669036626815796, 0.12880940735340118, 0.23446236550807953, 0.23033303022384644]
seed 42, draw 2 : [0.33669036626815796, 0.12880940735340118, 0.23446236550807953, 0.23033303022384644]
seed  7, draw 1 : [-0.1467950940132141, 0.7861412763595581, 0.9468216300010681, -1.1143440008163452]

same seed  -> identical? True
other seed -> different? True

Seeding works.


## 5. Run identity - the config hash

Every run will write one line to `results/runs.jsonl`. For that file to be
useful we need to know exactly *what* produced each line.

`RunConfig` holds every choice that affects the computation. Its `hash` is a
12-character fingerprint of those choices plus the seed. Two runs with the same
hash should produce the same numbers.

That fingerprint is what makes the full sweep **resumable**: `run_all.py` reads
the ids already in `runs.jsonl` and skips them, so a crashed 12-run matrix can
be restarted without redoing finished work.

In [6]:
from src.config import RunConfig

base = RunConfig(arm="A_transfer", target_domain="cora",
                 source_domains=("photo", "ppi", "elliptic"), seed=0)

print("run_id :", base.run_id)
print("hash   :", base.hash)

# Change only the seed -> different run, different hash.
other_seed = RunConfig(arm="A_transfer", target_domain="cora",
                       source_domains=("photo", "ppi", "elliptic"), seed=1)

# Change only the human note -> SAME computation, so the hash must not move.
with_note = RunConfig(arm="A_transfer", target_domain="cora",
                      source_domains=("photo", "ppi", "elliptic"), seed=0,
                      notes="rerun after fixing a typo in the README")

print()
print("different seed -> different hash?", base.hash != other_seed.hash)
print("different note -> same hash?     ", base.hash == with_note.hash)

assert base.hash != other_seed.hash
assert base.hash == with_note.hash
print("\nConfig fingerprinting behaves correctly.")

run_id : A_transfer__cora__lf1__s0__494221a92875
hash   : 494221a92875

different seed -> different hash? True
different note -> same hash?      True

Config fingerprinting behaves correctly.


## 6. Where things get written

Three locations, all resolved automatically:

- **data root** - downloaded datasets (~1 GB once notebook 01 has run)
- **results/runs.jsonl** - one JSON line per finished run
- **results/figures/** - every figure the notebooks save

The data root adapts to the machine: `$OMNIGRAPH_DATA` if you set it, else
Kaggle's writable layer, else Colab's, else `data/` in the project. You never
edit a path to move between them.

In [7]:
from src import config

config.ensure_dirs()

print("project root :", config.PROJECT_ROOT)
print("data root    :", config.DATA_ROOT)
print("results      :", config.RESULTS_DIR)
print("figures      :", config.FIGURES_DIR)
print("runs.jsonl   :", config.RUNS_JSONL)
print()
print("data root exists   :", config.DATA_ROOT.exists())
print("figures dir exists :", config.FIGURES_DIR.exists())
print("runs logged so far :", len(config.load_runs()))

project root : D:\documents\my projects\afterPFE\OmniGraph
data root    : D:\documents\my projects\afterPFE\OmniGraph\data
results      : D:\documents\my projects\afterPFE\OmniGraph\results
figures      : D:\documents\my projects\afterPFE\OmniGraph\results\figures
runs.jsonl   : D:\documents\my projects\afterPFE\OmniGraph\results\runs.jsonl

data root exists   : True
figures dir exists : True
runs logged so far : 0


## 7. Device check with a real tensor

`torch.cuda.is_available()` returning `True` is not quite proof. Let us
actually put a tensor on the device and do arithmetic on it.

In [8]:
import time

device = config.get_device()
print("selected device:", device)

x = torch.randn(2000, 2000, device=device)
start = time.perf_counter()
y = x @ x
if device.type == "cuda":
    torch.cuda.synchronize()   # GPU calls are async; wait before timing
elapsed = time.perf_counter() - start

print(f"2000x2000 matmul on {device}: {elapsed * 1000:.1f} ms")
print("result device:", y.device, "| shape:", tuple(y.shape), "| dtype:", y.dtype)

if device.type == "cuda":
    used = torch.cuda.memory_allocated() / 1024 ** 2
    total = torch.cuda.get_device_properties(0).total_memory / 1024 ** 2
    print(f"GPU memory in use: {used:.0f} MB of {total:.0f} MB")
    del x, y
    torch.cuda.empty_cache()

selected device: cuda


2000x2000 matmul on cuda: 135.3 ms
result device: cuda:0 | shape: (2000, 2000) | dtype: torch.float32
GPU memory in use: 40 MB of 4096 MB


## What you should now understand

- **Reproducibility is built, not assumed.** Three mechanisms carry it:
  pinned package versions, `set_seed` covering all four RNGs, and a config
  hash that ties every line in `results/runs.jsonl` back to the exact settings
  that produced it.
- **The code never hard-codes a path or a device.** `config.DATA_ROOT` and
  `config.get_device()` resolve themselves, which is what lets the identical
  notebook run on a laptop CPU, a laptop GPU, and a Kaggle T4.
- **The notebook is a presentation layer.** Everything printed above was
  computed in `src/`. Nothing here defines logic, so a script can re-derive
  the same facts later without copying a cell.

**Next:** `01_datasets.ipynb` downloads the four domains and asks what they
actually contain - which is where it becomes obvious how unalike they are.